# Mixed-Coordinate Arithmetic

A common pattern in energy modeling: variables cover **different subsets** of a shared dimension, but cost parameters span the full set. This notebook shows how to combine them cleanly under the v1 arithmetic convention.

**Scenario:** Three capacity variables (`cap_a`, `cap_b`, `cap_c`) cover different technology subsets. Cost coefficients are defined over all technologies. We want a single cost expression over the union of technologies.

In [ ]:
import xarray as xr

import linopy

linopy.options["arithmetic_convention"] = "v1"

## Setup

Three technology groups with overlapping cost data:

In [ ]:
m = linopy.Model()

tech_a = ["wind", "solar"]
tech_b = ["gas"]
tech_all = ["wind", "solar", "gas"]

cap_a = m.add_variables(lower=0, coords=[tech_a], dims=["tech"], name="cap_a")
cap_b = m.add_variables(lower=0, coords=[tech_b], dims=["tech"], name="cap_b")
cap_c = m.add_variables(lower=0, coords=[tech_all], dims=["tech"], name="cap_c")

# Cost parameters span all technologies — NaN where a variable doesn't apply
cost_a = xr.DataArray([7, 9, float("nan")], coords=[("tech", tech_all)])
cost_b = xr.DataArray([float("nan"), float("nan"), 11], coords=[("tech", tech_all)])
cost_c = xr.DataArray([13, 17, 19], coords=[("tech", tech_all)])

## Approach 1: `fillna(0)` + explicit joins

The most explicit approach. Since `cost_a` has NaN at "gas" (where `cap_a` doesn't exist), fill NaN with 0 before multiplying. Use `join="left"` so the product keeps only the variable's coordinates, then `join="outer"` when adding to build the union.

`fillna(0)` on a cost means "this technology has no cost contribution from this variable" — a safe, intentional choice.

In [ ]:
combined = (
    cap_a.mul(cost_a.fillna(0), join="left")
    .add(cap_b.mul(cost_b.fillna(0), join="left"), join="outer")
    .add(cap_c.mul(cost_c, join="left"), join="outer")
)
combined

Expected result:
```
[gas]:   +11 cap_b[gas]   + 19 cap_c[gas]
[solar]: +9  cap_a[solar] + 17 cap_c[solar]
[wind]:  +7  cap_a[wind]  + 13 cap_c[wind]
```

## Approach 2: `dropna()` on costs first

Instead of filling NaN with 0, drop the irrelevant entries from the cost arrays. Then multiply with `join="left"` (the variable's coords are always a subset of the cost's coords after dropping), and combine with `join="outer"`.

In [ ]:
combined_v2 = (
    cap_a.mul(cost_a.dropna("tech"), join="left")
    .add(cap_b.mul(cost_b.dropna("tech"), join="left"), join="outer")
    .add(cap_c.mul(cost_c, join="left"), join="outer")
)
combined_v2

## Approach 3: Scope costs to each variable upfront

The cleanest option when you control the data: define costs only over the relevant technologies from the start, eliminating NaN entirely.

In [ ]:
# Costs scoped to each variable's technologies — no NaN needed
cost_a_scoped = xr.DataArray([7, 9], coords=[("tech", tech_a)])
cost_b_scoped = xr.DataArray([11], coords=[("tech", tech_b)])
cost_c_scoped = xr.DataArray([13, 17, 19], coords=[("tech", tech_all)])

combined_v3 = (
    (cap_a * cost_a_scoped)
    .add(cap_b * cost_b_scoped, join="outer")
    .add(cap_c * cost_c_scoped, join="outer")
)
combined_v3

## Approach 4: Pre-align with `linopy.align()`

Use `linopy.align()` to reindex all variables and cost arrays to the same coordinates upfront. After alignment, all operands share the same `tech` dimension, so arithmetic uses exact matching — no per-operation `join=` needed.

Variables get absent slots at coordinates they don't cover; cost arrays get NaN. Since NaN in a multiplicative constant acts as a mask, the NaN entries naturally produce absent terms — no `fillna` needed.

In [ ]:
# Align all variables and costs to the union of tech coordinates
cap_a_al, cap_b_al, cap_c_al, cost_a_al, cost_b_al, cost_c_al = linopy.align(
    cap_a, cap_b, cap_c, cost_a, cost_b, cost_c, join="outer"
)

# NaN in costs naturally masks — no fillna needed!
combined_v4 = cap_a_al * cost_a_al + cap_b_al * cost_b_al + cap_c_al * cost_c_al
combined_v4

---

## Adding a partial scaling factor

Now extend the example: a `rate` parameter applies only to gas technologies. We want `cap_c * cost_c * rate`, where `rate` defaults to 1 for technologies it doesn't cover.

Expected result — same as before, but the gas entry for `cap_c` is scaled by 1.04:
```
[gas]:   +11 cap_b[gas]   + 19.76 cap_c[gas]
[solar]: +9  cap_a[solar] + 17    cap_c[solar]
[wind]:  +7  cap_a[wind]  + 13    cap_c[wind]
```

In [ ]:
rate = xr.DataArray([1.04], coords=[("tech", ["gas"])])

### Option A: `fill_value=1` on `.mul()`

The `fill_value` parameter tells linopy what to use for technologies not covered by `rate`. Since `rate` is a scaling factor, `1` is the natural identity — "no scaling".

In [ ]:
combined_rate_a = (
    cap_a.mul(cost_a.fillna(0), join="left")
    .add(cap_b.mul(cost_b.fillna(0), join="left"), join="outer")
    .add(
        cap_c.mul(cost_c, join="left").mul(rate, join="left", fill_value=1),
        join="outer",
    )
)
combined_rate_a

### Option B: Prepare the parameter with xarray first

Pre-multiply the cost and rate arrays using standard xarray operations before passing to linopy. This keeps the linopy arithmetic simple.

In [ ]:
# Extend rate to all techs (fill with 1 = no scaling), then multiply with cost
cost_c_rated = cost_c * rate.reindex(tech=tech_all).fillna(1)
print("cost_c_rated:", cost_c_rated.values)  # [13, 17, 19.76]

combined_rate_b = (
    cap_a.mul(cost_a.fillna(0), join="left")
    .add(cap_b.mul(cost_b.fillna(0), join="left"), join="outer")
    .add(cap_c * cost_c_rated, join="outer")
)
combined_rate_b

---

## NaN as mask

Under the v1 convention, NaN in a multiplicative constant **masks** the corresponding term — the position becomes absent. This means you can use NaN-containing cost arrays directly with `join="left"` and the NaN entries will naturally drop out:

In [ ]:
# NaN in cost_a at "gas" naturally masks cap_a at "gas" — no fillna needed!
combined_nan_mask = (
    cap_a.mul(cost_a, join="left")
    .add(cap_b.mul(cost_b, join="left"), join="outer")
    .add(cap_c * cost_c, join="outer")
)
combined_nan_mask

---

## Summary of patterns

| Situation | Solution |
|---|---|
| Cost array has NaN for irrelevant techs | NaN in mul/div **masks** the term (makes it absent) — no cleanup needed |
| NaN in additive constant | NaN treated as 0 (additive identity) — no cleanup needed |
| Variables have different coord subsets | Use `.add(..., join="outer")` to build the union |
| Pre-align all operands | `linopy.align(*vars, *costs, join="outer")`, then use `+`/`*` directly |
| Multiplication with matching coords | `var * cost` (exact match, no join needed) |
| Multiplication with superset cost | `var.mul(cost, join="left")` to keep var's coords |
| Partial scaling factor (e.g., rate for some techs) | `expr.mul(rate, join="left", fill_value=1)` |
| Partial scaling factor (alternative) | Pre-compute `cost * rate.reindex_like(cost).fillna(1)` in xarray |

**Key principle:** NaN in multiplicative constants means "no term here" (absent). NaN in additive constants means "no contribution" (zero). For scaling factors where missing means "identity," use `fill_value=1` explicitly.